In [5]:
import json
from typing import Dict, Any, List


class NotebookParseError(Exception):
    """Custom exception raised when notebook parsing fails due to invalid structure."""
    pass

def parse_ipynb(file_path) -> Dict[str, List[Dict[str, Any]]]:
        """
        Parses a Jupyter Notebook (.ipynb) file and extracts markdown and code cells.

        Raises:
            FileNotFoundError: If the path does not exist.
            PermissionError: If read permissions are lacking.
            ValueError: If the file is not valid JSON or UTF-8.
            NotebookParseError: If the JSON structure does not match a standard notebook schema.
        """
        if not file_path:
            raise ValueError("FILE DOES NOT EXIST")

        # 1. File Access & Read Handling
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                notebook = json.load(f)
        except FileNotFoundError:
            raise FileNotFoundError(f"File not found: '{file_path}'")
        except PermissionError:
            raise PermissionError(f"Permission denied when reading '{file_path}'")
        except UnicodeDecodeError as e:
            raise ValueError(
                f"File '{file_path}' is not encoded in valid UTF-8 format. Details: {e}"
            )
        except json.JSONDecodeError as e:
            raise ValueError(
                f"Invalid notebook file. '{file_path}' is not valid JSON. Details: {e}"
            )

        # 2. Schema Validation
        if not isinstance(notebook, dict):
            raise NotebookParseError(
                "Malformed notebook: Root structure must be a JSON object."
            )

        if "cells" not in notebook or not isinstance(notebook["cells"], list):
            raise NotebookParseError(
                "Malformed notebook: Missing or invalid 'cells' list."
            )

        parsed_data: Dict[str, List[Dict[str, Any]]] = {"markdown": [], "code": []}

        # 3. Safe Cell Extraction
        for index, cell in enumerate(notebook["cells"]):
            if not isinstance(cell, dict):
                continue  # Skip invalid non-dict cell entries

            cell_type = cell.get("cell_type")
            source = cell.get("source", "")

            # Safely normalize 'source' into a string
            if isinstance(source, list):
                source = "".join(str(line) for line in source)
            elif not isinstance(source, str):
                source = str(source)

            cell_details = {
                "cell_index": index,
                "source": source,
                "metadata": (
                    cell.get("metadata", {})
                    if isinstance(cell.get("metadata"), dict)
                    else {}
                ),
                "id": cell.get("id"),
            }

            if cell_type == "markdown":
                parsed_data["markdown"].append(cell_details)

            elif cell_type == "code":
                cell_details["execution_count"] = cell.get("execution_count")
                cell_details["outputs"] = (
                    cell.get("outputs", [])
                    if isinstance(cell.get("outputs"), list)
                    else []
                )
                parsed_data["code"].append(cell_details)

        return parsed_data

# Usage:
# result = parse_ipynb("example_notebook.ipynb")
# print(json.dumps(result, indent=2))

In [7]:
print(parse_ipynb("/home/user/Documents/Project_5/backend/experiments/dependencies_parser.ipynb"))

{'markdown': [{'cell_index': 0, 'source': '## This is going to be a dependencies parser for JS, i.e, package.json\n\nThe objective is to get this structure\n~~~\n{\n    "details": {\n        "name": "name of the project",\n        "version": "version of the project",\n        .... and other details from the package.json\n    },\n    "dependencies": [{\n        "name": "dep name",\n        "version": "dep version"\n        "type" : "dev or so on"\n    }, ....\n    ]\n}\n~~~', 'metadata': {}, 'id': 'a6b93854'}], 'code': [{'cell_index': 1, 'source': '', 'metadata': {}, 'id': '2e317e4f', 'execution_count': None, 'outputs': []}, {'cell_index': 2, 'source': 'import json\nfrom pathlib import Path\nfrom typing import Union\n\n\ndef parse_package_json(file_path: Union[str, Path]) -> dict:\n    file_path = Path(file_path)\n\n    if not file_path.exists():\n        raise FileNotFoundError(f"File not found: {file_path}")\n\n    try:\n        with open(file_path, "r", encoding="utf-8") as f:\n     